In [ ]:
function TrAdaBoost_with_LabelProportionLoss(SourceData, TargetData, MaxRounds, Lambda):
    # Input: 
    # - SourceData: (X_S, y_S) with N_S samples
    # - TargetData: (X_T, y_T) with N_T samples
    # - MaxRounds: Number of boosting rounds
    # - Lambda: Weight for Label Proportion Loss

    Initialize weights for source samples: w_S = [1 / N_S] * N_S
    Initialize weights for target samples: w_T = [1 / N_T] * N_T

    for round in 1 to MaxRounds do:
        # Step 1: Train Weak Learner
        Train weak learner h_t using combined dataset (SourceData, TargetData) weighted by w_S and w_T

        # Step 2: Compute Classification Error
        Compute weighted classification error on target samples:
        error_T = \sum_{i \in TargetData} w_T[i] * I(h_t(X_T[i]) != y_T[i])

        # Avoid division by zero or overfitting:
        if error_T >= 0.5 then break

        # Step 3: Update Target Weights
        for each target sample i in TargetData do:
            if h_t(X_T[i]) == y_T[i]:
                w_T[i] = w_T[i] * (error_T / (1 - error_T))
        Normalize w_T

        # Step 4: Estimate Label Proportions
        Predicted label proportions on target domain:
        P_T = EstimateLabelProportions(h_t, TargetData)

        True label proportions from source domain:
        P_S = ComputeTrueLabelProportions(SourceData)

        # Step 5: Compute Label Proportion Loss
        LabelProportionLoss = KL_Divergence(P_T, P_S)

        # Step 6: Update Source Weights
        for each source sample i in SourceData do:
            Adjust weight based on classification loss and contribution to LabelProportionLoss:
            if h_t(X_S[i]) == y_S[i]:
                w_S[i] = w_S[i] * exp(-Lambda * LabelProportionLoss)
            else:
                w_S[i] = w_S[i] * exp(Lambda * LabelProportionLoss)
        Normalize w_S

    end for

    # Final Model: Weighted ensemble of weak learners
    return FinalModel = CombineWeakLearners(h_1, h_2, ..., h_t, weights)

# Helper Functions
function EstimateLabelProportions(h_t, TargetData):
    P_T = [0] * num_classes
    for each sample (X_T[i], y_T[i]) in TargetData do:
        P_T[h_t(X_T[i])] += 1 / N_T
    return Normalize(P_T)

function ComputeTrueLabelProportions(SourceData):
    P_S = [0] * num_classes
    for each sample (X_S[i], y_S[i]) in SourceData do:
        P_S[y_S[i]] += 1 / N_S
    return Normalize(P_S)

function KL_Divergence(P_T, P_S):
    return \sum_{c} P_T[c] * log(P_T[c] / (P_S[c] + epsilon))  # Avoid division by zero

function CombineWeakLearners(h_1, h_2, ..., h_t, weights):
    return WeightedEnsemble(h_1, h_2, ..., h_t, weights)

# Example Usage
FinalModel = TrAdaBoost_with_LabelProportionLoss(SourceData, TargetData, MaxRounds=50, Lambda=0.1)
